# Normalization for Identity Resolution Phase: Wikidata

This notebook normalizes the Wikidata airport dataset (`wikidataMapped.xml`) for the identity resolution phase.

Steps applied:

0. Country columns: add `countryCode` from `countryName` via pycountry fuzzy search
1. Lowercase & remove punctuation
2. Strip spaces
3. Unicode normalization (replace ï, ö, ä, accented chars)
4. Translation *(SKIPPED – pending coaching session)*
5. Abbreviation expansion
6. Remove stopwords and common airport words


In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
from PyDI.io import load_xml

DATA_DIR = Path("../../schema/wikidata")
OUTPUT_DIR = Path("../../data/normalized_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

XML_FILE = DATA_DIR / "wikidataMapped.xml"

df = load_xml(XML_FILE, name="wikidata")
df.attrs['source'] = 'wikidata'

print(f"Loaded {len(df)} records")
print(f"Columns: {list(df.columns)}")
df.tail(5)

Loaded 24859 records
Columns: ['id', 'name', 'iata', 'icao', 'lat', 'lon', 'elevation', 'countryName', 'city', 'wikidataURI', 'operator', 'timezone', 'owner', 'localCode']


,id,name,iata,icao,lat,lon,elevation,countryName,city,wikidataURI,operator,timezone,owner,localCode
24854,wd_24855,Krasnoshchelye,NaN,ULMX,67.3477088449,37.07824574511,NaN,Russia,Lovozersky District,http://www.wikidata.org/entity/Q99746020,NaN,NaN,NaN,NaN
24855,wd_24856,Umba,NaN,ULMA,66.69639289071,34.38557611255,NaN,Russia,Tersky District,http://www.wikidata.org/entity/Q99746045,NaN,NaN,NaN,NaN
24856,wd_24857,Hazrat Sultan International Airport,HSA,NaN,43.31111100811,68.55027784458,NaN,Kazakhstan,Kazakhstan,http://www.wikidata.org/entity/Q99776256,NaN,UTC+05:00,NaN,NaN
24857,wd_24858,Santarém Airport,NaN,LPSR,39.20944445693,-8.687222216918,NaN,Portugal,Portugal,http://www.wikidata.org/entity/Q998862,NaN,UTC±00:00,NaN,NaN
24858,wd_24859,Jorge Newbery Airport,AEP,SABE,-34.55888880841,-58.41638892742,5,Argentina,Buenos Aires,http://www.wikidata.org/entity/Q999047,Aeropuertos Argentina,NaN,NaN,NaN


## 0. Country columns – add `countryCode`

### Pycountry
Wikidata has **only `countryName`** (no ISO code). We first try the `pycountry` package's fuzzy search method to derive `countryCode`:

```python
import pycountry
results = pycountry.countries.search_fuzzy("People's Republic of China")
# → [Country(alpha_2='CN', name='China', ...)]
```

`search_fuzzy` returns a **list** — always take the first result (`[0]`). Wrap in try/except to handle names that cannot be matched.

This allows `countryCode` to be used as a blocking key across all three datasets.

This step should be completed before the normalization of the `countryName` column, as the fuzzy search algorithm is not strong enough to find normalized values.

In [2]:
import pycountry

def pycountry_search_code(country_name):
    if not isinstance(country_name, str):
        return None
    try:
        candidates = pycountry.countries.search_fuzzy(country_name)
        return candidates[0].alpha_2
    except LookupError:
        return None

# To avoid repetitive search, we build a dict {distinctCountryName: countryCode}, fuzzy search for each country name, then apply the map to the dataframe.
pycountry_map = {
    c: pycountry_search_code(c)
    for c in df['countryName'].dropna().unique()
}

In [3]:
# print countries that pycountry failed to search for the codes
print([n for n, c in pycountry_map.items() if c is None])

['Turkey', 'Empire of Japan', 'Soviet Union', 'Democratic Republic of the Congo', 'Mandatory Palestine', 'Israel and The Occupied Territories', 'Ivory Coast', 'Cape Verde', 'Somaliland', 'British Raj', "Luhansk People's Republic", 'Republic of Abkhazia', "Great Socialist People's Libyan Arab Jamahiriya", 'Nazi Germany', 'Ethiopian Empire', 'Trucial States', 'German Reich']


In [4]:
for c in ['Niger', 'Republic of China', 'Kosovo', 'Sint Maarten', 'Curaçao', 'British India']:
    print(f"{c:<24} {pycountry_map[c]}")

Niger                    NG
Republic of China        CN
Kosovo                   RS
Sint Maarten             NL
Curaçao                  NL
British India            IO


For the countries above, `pycountry` returns incorrect country code.
For example, "Republic of China" is actually Taiwan, so the country code is supposed to be `TW`. In addition, "Niger"'s correct ISO code is NE, not NG (Nigeria).


### Countryguess

We considered another alternative: the `countryguess` package to search for the country code. 
```python
from countryguess import guess_country
guess_country("TW", attribute="name_official")  # 2-letter code lookup
# -> Republic of China
```

`countryguess` uses human-maintained regex rules for every country in its database, so we expect it to be more robust than `pycountry`'s substring search. Only when no rule matches does it fall back to string similarity, which is where its remaining errors come from.

In [5]:
from countryguess import guess_country

countryguess_map = {
    c: guess_country(c, attribute="iso2")   # returns None when not found
    for c in df['countryName'].dropna().unique()
}

# print countries that countryguess failed to search for the codes
print([n for n, c in countryguess_map.items() if c is None])

['British Raj', "Luhansk People's Republic", 'German Reich']


We can see that unmatched countries are fewer than those for `pycountry`.

In [6]:
for c in ['Niger', 'Republic of China', 'Kosovo', 'Sint Maarten', 'Curaçao', 'British India']:
    print(f"{c:<24} {countryguess_map[c]}")

Niger                    NE
Republic of China        TW
Kosovo                   XK
Sint Maarten             SX
Curaçao                  CW
British India            IN


Most countries that `pycountry` answers incorrectly are mapped correctly by `countryguess`. However, after diving more deeply, we found that:

1. Historical states (e.g., Nazi Germany, Soviet Union, Ethiopian Empire) have changing borders, so the airport may now lie in a different country than the nominal successor state. We therefore leave countryCode null for these records.
2. `countryguess` still returns a wrong code for a few names (e.g., Republic of Abkhazia → AL, Trucial States → OM).

In [7]:
HISTORICAL_COUNTRIES = {"Ethiopian Empire", "Nazi Germany", "Soviet Union", "Empire of Japan", "British India", "Mandatory Palestine"}
INCORRECT_ITEMS = {"Trucial States", "Republic of Abkhazia"}

for c in HISTORICAL_COUNTRIES | INCORRECT_ITEMS:
    print(f"{c:<40} {countryguess_map[c]}")

Trucial States                           OM
British India                            IN
Republic of Abkhazia                     AL
Ethiopian Empire                         ET
Nazi Germany                             DE
Empire of Japan                          JP
Mandatory Palestine                      PS
Soviet Union                             SU


Remove `HISTORICAL_COUNTRIES` and `INCORRECT_ITEMS` from the map.

In [8]:
for c in HISTORICAL_COUNTRIES | INCORRECT_ITEMS:
    countryguess_map[c] = None

### Map countryguess search result to the whole DataFrame

In [9]:
df['countryCode'] = df['countryName'].map(countryguess_map)

print(f"{df['countryCode'].isna().sum()} records are left without countryCode. Leave them for identity resolution.")


31 records are left without countryCode. Leave them for identity resolution.


### Finding: Namibia's ISO code `NA` is lost as a missing value

Namibia's ISO 3166-1 alpha-2 code is the string `"NA"`, which `pandas.read_csv` treats as a missing value by default.
This affects the other two sources:

- **Kaggle**: the raw file already has an empty `Country_CodeA2` for all 14 Namibia records (`Country_CodeA3` is still `NAM`). To restore it, set `countryCode = "NA"` where `countryName == "Namibia"`.
- **OurAirports**: the raw file contains `NA` for 303 records, but `pd.read_csv` in `ourairports_data_cleaning.ipynb` converted them to NaN, so the interim CSV and `OurAirports.xml` have 303 records without `countryCode`. Schema mapping does not need to be redone; when the field is needed, fill the missing `countryCode` from the prefix of `isoRegion` (e.g. `NA-KA` -> `NA`). The counts were verified: 303 raw `NA` values = 303 XML records without `countryCode`, all with `isoRegion` starting with `NA-`.

This applies to all three sources: once Namibia's `NA` is restored (Wikidata via `countryguess`, Kaggle and OurAirports via the fixes above), every step that reads a CSV with pandas must use `keep_default_na=False` and `na_values=[""]`, otherwise `NA` is turned into a missing value again. Reading XML with `PyDI.io.load_xml` is not affected (verified with a test record).

Verification script:

In [10]:
# OurAirports: raw 'NA' count vs XML records without countryCode
print("--- OurAirports ---")
raw = pd.read_csv("../../data/raw/ourairports/airports-ourairports.csv",
                  keep_default_na=False)          # keep "NA" as a string
n_raw_na = (raw["iso_country"] == "NA").sum()
n_raw_empty = (raw["iso_country"] == "").sum()

oa = load_xml("../../schema/ourairports/OurAirports.xml", name="ourairports")
missing = oa[oa["countryCode"].isna()]
prefix_ok = missing["isoRegion"].fillna("").str.startswith("NA-")

print(f"raw 'NA': {n_raw_na} | raw empty: {n_raw_empty}")
print(f"XML without countryCode: {len(missing)} | of which isoRegion starts with 'NA-': {prefix_ok.sum()}")
# expected: 303 | 0 / 303 | 303

# Kaggle: records without countryCode should all be Namibia
print("\n--- Kaggle ---")
kg = load_xml("../../schema/kaggle/kaggleMapped.xml", name="kaggle")
print(kg[kg["countryCode"].isna()]["countryName"].value_counts())
# expected: Namibia 14


--- OurAirports ---
raw 'NA': 303 | raw empty: 0
XML without countryCode: 303 | of which isoRegion starts with 'NA-': 303

--- Kaggle ---
countryName
Namibia    14
Name: count, dtype: int64


In [11]:
assert set(df.loc[df["countryName"] == "Namibia", "countryCode"]) == {"NA"}

## 1. Lowercase & remove punctuation for string columns

Targeted columns: `name`, `countryName`, `city`

Removes: commas, dots, parentheses, `'`, `"`, hyphens (kept as space), and other punctuation.

In [12]:
STRING_COLS = ["name", "countryName", "city"]

# Apply the same rule as in Kaggle
# Characters to remove completely
REMOVE_CHARS = re.compile(r"[,\.\(\)'\"\!\?\:\;\`\~‘’“”]") 
HYPHEN_CHARS = re.compile(r"[-/&‐‑–—]")

def lowercase_remove_punct(text):
    if not isinstance(text, str):
        return text
    text = text.lower()
    text = HYPHEN_CHARS.sub(" ", text)
    text = REMOVE_CHARS.sub("", text)
    return text

for col in STRING_COLS:
    if col in df.columns:
        df[col] = df[col].apply(lowercase_remove_punct)

print("Sample after step 1:")
df[["id", "name", "countryName", "city"]].tail(10)

Sample after step 1:


,id,name,countryName,city
24849,wd_24850,václav havel airport prague,czech republic,ruzyně
24850,wd_24851,lincang airport,peoples republic of china,yunnan
24851,wd_24852,esztergom airport,hungary,esztergom
24852,wd_24853,cherbourg maupertus airport,france,gonneville le theil
24853,wd_24854,araxos airport,greece,west achaea municipality
24854,wd_24855,krasnoshchelye,russia,lovozersky district
24855,wd_24856,umba,russia,tersky district
24856,wd_24857,hazrat sultan international airport,kazakhstan,kazakhstan
24857,wd_24858,santarém airport,portugal,portugal
24858,wd_24859,jorge newbery airport,argentina,buenos aires


## 2. Strip spaces

Remove leading/trailing whitespace and collapse multiple internal spaces.

In [13]:
MULTI_SPACE = re.compile(r"\s+")

def strip_spaces(text):
    if not isinstance(text, str):
        return text
    return MULTI_SPACE.sub(" ", text).strip()

for col in STRING_COLS:
    if col in df.columns:
        df[col] = df[col].apply(strip_spaces)

# Show stripped result
print("Sample after step 2:")
df[["id", "name"]].head(10)

Sample after step 2:


,id,name
0,wd_1,jian jinggangshan airport
1,wd_2,changde taohuayuan airport
2,wd_3,jiujiang lushan airport
3,wd_4,aksu hongqipo airport
4,wd_5,mohe gulian airport
5,wd_6,surbung airport
6,wd_7,dehong mangshi airport
7,wd_8,panzhihua baoanying airport
8,wd_9,quzhou airport
9,wd_10,taizhou luqiao airport


## 3. Unicode normalization

Uses NFKD decomposition to separate base characters from diacritics, then encodes to ASCII (ignoring the diacritic combining characters).

Examples:
- ï, ö, ä → i, o, a
- São Paulo → sao paulo
- Piñon → pinon
- Çarşamba → carsamba

In [14]:
def normalize_unicode(text):
    if not isinstance(text, str):
        return text
    # NFKD decomposes combined chars; encode/decode drops combining diacritics
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")

for col in STRING_COLS:
    if col in df.columns:
        df[col] = df[col].apply(normalize_unicode)

# strip spaces again after normilzing the unicodes
for col in STRING_COLS:
    if col in df.columns:
        df[col] = df[col].apply(strip_spaces)

print("\nSample after step 3:")
df[["id", "name", "countryName"]].tail(10)


Sample after step 3:


,id,name,countryName
24849,wd_24850,vaclav havel airport prague,czech republic
24850,wd_24851,lincang airport,peoples republic of china
24851,wd_24852,esztergom airport,hungary
24852,wd_24853,cherbourg maupertus airport,france
24853,wd_24854,araxos airport,greece
24854,wd_24855,krasnoshchelye,russia
24855,wd_24856,umba,russia
24856,wd_24857,hazrat sultan international airport,kazakhstan
24857,wd_24858,santarem airport,portugal
24858,wd_24859,jorge newbery airport,argentina


## 4. Translation *(SKIPPED FOR NOW: pending coaching session)*

Wikidata has many airports with non-English names. Discuss with professor whether translation is worthwhile.

## 5. Abbreviation expansion

The Wikidata dataset uses various abbreviations in the `name` column. The abbreviations below were identified by:
1. The known list from the task description
2. A frequency analysis of title-case and all-caps short tokens in the name column

Expansion is applied as **whole-word** substitution (word boundaries `\b`) to avoid accidental replacements inside longer words.

In [15]:
# Abbreviation map (lowercase key -> expanded form, also lowercase)
# Order matters for overlapping patterns – more specific first
ABBREVIATION_MAP = [
    # Directly from task description
    (r"\bintl\b",  "international"),
    (r"\brgnl\b",  "regional"),
    (r"\bmuni\b",  "municipal"),
    (r"\bafb\b",   "air force base"),
    (r"\bfld\b",   "field"),
    # st: keep as 'saint' in airport context ("St. Louis" -> saint louis)
    # street is much less common in airport names than saint
    (r"\bst\.?\b", "saint"),
    # Additional abbreviations found via frequency analysis
    (r"\baaf\b",   "army air field"),
    (r"\bnas\b",   "naval air station"),
    (r"\bab\b",    "air base"),
    (r"\bstn\b",   "station"),
    # * In wikidata, co does not only refer to 'county'. For example, 
    # * cutral co airport → cutral county airport: 'Có' in Cutral Có refers to a city in Argentina, not 'county'. So we do not include this rule to Wikidata.
    # (r"\bco\.?\b", "county"),
    (r"\bmem\b",   "memorial"),
    (r"\blrrs\b",  "long range radar site"),
    (r"\bhwy\b",   "highway"),
    (r"\bmt\.?\b", "mount"),
    (r"\bft\.?\b", "fort"),
    (r"\bpt\.?\b", "point"),
    (r"\bjct\.?\b","junction"),
    (r"\bblvd\.?\b","boulevard"),
]

# Compile patterns once
COMPILED_ABBREVS = [
    (re.compile(pattern, re.IGNORECASE), expansion)
    for pattern, expansion in ABBREVIATION_MAP
]

def expand_abbreviations(text):
    if not isinstance(text, str):
        return text
    for pattern, expansion in COMPILED_ABBREVS:
        text = pattern.sub(expansion, text)
    return text


# Show real Wikidata names matched by each abbreviation pattern
names_before = df["name"].dropna().astype(str)

print("Abbreviations appearing in Wikidata:")
hits_ls = []
for pattern, expansion in COMPILED_ABBREVS:
    hits = names_before[names_before.str.contains(pattern, regex=True)]
    if hits.empty:
        continue
    hits_ls.append((pattern, hits))
hits_ls.sort(key=lambda x: len(x[1]), reverse=True)

print(f"{'pattern':<14}{'hits':<8}example")
for pattern, hits in hits_ls:
    original = hits.iloc[0]
    print(f"{pattern.pattern:<14}{len(hits):<8}{original} -> {expand_abbreviations(original)}")




# Apply expand_abbreviations to all names
# Only apply to 'name' column (abbreviations appear there, not in country/city)
df["name"] = df["name"].apply(expand_abbreviations)

# Strip spaces again after expansion (expansions may create multiple spaces)
df["name"] = df["name"].apply(strip_spaces)


print("\nSample after step 5:")
df[["id", "name"]].tail(15)

Abbreviations appearing in Wikidata:
pattern       hits    example
\bst\.?\b     80      st clair county international airport -> saint clair county international airport
\bmuni\b      8       bowbells muni aerodrome -> bowbells municipal aerodrome
\blrrs\b      7       barter island lrrs airport -> barter island long range radar site airport
\bfld\b       6       idlers fld airport -> idlers field airport
\bab\b        6       vandel ab airport -> vandel air base airport
\bintl\b      2       freeflight intl airport -> freeflight international airport
\bnas\b       1       nas shibpur -> naval air station shibpur
\bmt\.?\b     1       mt hope airport -> mount hope airport
\bpt\.?\b     1       pt enterprise d w ranch airport -> point enterprise d w ranch airport

Sample after step 5:


,id,name
24844,wd_24845,bozhou airport
24845,wd_24846,antonio narino airport
24846,wd_24847,flughafen flugplatzstrae 100 in furth
24847,wd_24848,baoshan yunrui airport
24848,wd_24849,casas de juan gil airfield
24849,wd_24850,vaclav havel airport prague
24850,wd_24851,lincang airport
24851,wd_24852,esztergom airport
24852,wd_24853,cherbourg maupertus airport
24853,wd_24854,araxos airport


## 6. Remove stopwords and common words

After abbreviation expansion many names still contain common words like `international`, `airport`, `the`, `of`, etc. that add noise to string comparators.

Strategy: remove English function words and Romance/Germanic articles from the `name` column.

In [16]:
# Common English function words (equivalent to sklearn's ENGLISH_STOP_WORDS subset)
# Defined inline to avoid importing sklearn/scipy (binary incompatibility on some macOS builds)
ENGLISH_STOP_WORDS = frozenset([
    "a", "an", "the", "and", "or", "but", "if", "in", "on", "at", "to",
    "of", "for", "by", "with", "as", "is", "it", "its", "be", "are", "was",
    "were", "been", "have", "has", "had", "do", "does", "did", "not", "no",
    "nor", "so", "yet", "both", "either", "neither", "than", "that", "this",
    "these", "those", "which", "who", "whom", "when", "where", "how", "all",
    "each", "every", "any", "some", "such", "from", "into", "through",
    "during", "before", "after", "above", "below", "between", "out", "off",
    "over", "under", "again", "further", "then", "once", "there", "here",
])

# Extended with Romance/Germanic articles & prepositions
STOPWORDS = ENGLISH_STOP_WORDS | {
    "de", "la", "le", "el", "al", "del", "des", "les", "du", "da", "di", "der", "das",
}

STOPWORD_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(w) for w in sorted(STOPWORDS, key=len, reverse=True)) + r")\b"
)

def remove_stopwords(text):
    if not isinstance(text, str):
        return text
    text = STOPWORD_PATTERN.sub("", text)
    return strip_spaces(text)

# Keep original name before stopword removal for inspection
df["name_before_sw"] = df["name"].copy()
df["name"] = df["name"].apply(remove_stopwords)

changed = df[df["name"] != df["name_before_sw"]][["id", "name_before_sw", "name"]].head(20)
print(f"Stopword removal changed {len(df[df['name'] != df['name_before_sw']])} records")
display(changed)

df.drop(columns=["name_before_sw"], inplace=True)

Stopword removal changed 919 records


,id,name_before_sw,name
52,wd_53,manuel marquez de leon international airport,manuel marquez leon international airport
88,wd_89,mar de cortes international airport,mar cortes international airport
169,wd_170,airport of dekese,airport dekese
231,wd_232,ras al hadd airport,ras hadd airport
238,wd_239,saint georges de loyapock airport,saint georges loyapock airport
270,wd_271,prairie du chien municipal airport,prairie chien municipal airport
287,wd_288,la pedrera airport,pedrera airport
333,wd_334,university of oklahoma westheimer airport,university oklahoma westheimer airport
439,wd_440,chaumont aerodrome la vendue,chaumont aerodrome vendue
481,wd_482,mambramo raya a airport,mambramo raya airport


## Final: Save normalized dataset

In [17]:
output_csv = OUTPUT_DIR / "wikidataNormalized.csv"
df.to_csv(output_csv, index=False)
print(f"Saved {len(df)} normalized records to {output_csv}")

# Summary stats
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df[["id", "name", "countryCode", "countryName", "city"]].head(15)

Saved 24859 normalized records to ../../data/normalized_outputs/wikidataNormalized.csv

Dataset shape: (24859, 15)
Columns: ['id', 'name', 'iata', 'icao', 'lat', 'lon', 'elevation', 'countryName', 'city', 'wikidataURI', 'operator', 'timezone', 'owner', 'localCode', 'countryCode']


,id,name,countryCode,countryName,city
0,wd_1,jian jinggangshan airport,CN,peoples republic of china,taihe county
1,wd_2,changde taohuayuan airport,CN,peoples republic of china,hunan
2,wd_3,jiujiang lushan airport,CN,peoples republic of china,jiujiang
3,wd_4,aksu hongqipo airport,CN,peoples republic of china,hongqipo
4,wd_5,mohe gulian airport,CN,peoples republic of china,heilongjiang
5,wd_6,surbung airport,MM,myanmar,falam township
6,wd_7,dehong mangshi airport,CN,peoples republic of china,mangshi city
7,wd_8,panzhihua baoanying airport,CN,peoples republic of china,dongqu
8,wd_9,quzhou airport,CN,peoples republic of china,zhejiang
9,wd_10,taizhou luqiao airport,CN,peoples republic of china,luqiao district
